In [1]:
import pandas as pd

df_inp = pd.read_csv(r"D:\Coding\Data\Lanzhou_cfdc\processed\N_INP(202409-202509)v2.4.2.csv")
#df_inp = df_inp[df_inp['Is_Significant'] == True]

target_temp = -30
df_temp = df_inp[df_inp['T_a(degC)'] == target_temp]
conc_mean = df_temp['N_INP(#/L)'].groupby(df_temp['Season']).mean()
conc_std = df_temp['N_INP(#/L)'].groupby(df_temp['Season']).std()
conc_count = df_temp['N_INP(#/L)'].groupby(df_temp['Season']).count()
print(conc_mean)
print(conc_std)
print(conc_count)

Season
Autumn    51.633631
Spring    84.004129
Summer    23.410065
Winter     5.695179
Name: N_INP(#/L), dtype: float64
Season
Autumn     70.364400
Spring    102.218263
Summer     19.891953
Winter      5.259645
Name: N_INP(#/L), dtype: float64
Season
Autumn    299
Spring    538
Summer    211
Winter    300
Name: N_INP(#/L), dtype: int64


In [2]:
df_temp

,Time,N_INP(#/L),T_INP(degC),SS_w,SS_i,Total_Sampling_Volume(L),Significance_Level(#/L),Is_Significant,Season,T_a(degC)
0,2024-09-24 14:41:43.068586000,14.050137,-29.797332,5.761454,41.444960,6.149300,2.617877,True,Autumn,-30
1,2024-09-26 14:13:58.060174000,31.918336,-29.789597,5.983611,41.736518,6.291500,4.248877,True,Autumn,-30
2,2024-09-26 14:26:14.060563000,27.829487,-29.781749,5.968580,41.693702,6.266200,4.178253,True,Autumn,-30
3,2024-09-26 16:07:50.060424000,15.449528,-29.756955,4.246887,39.362124,6.032083,4.010179,True,Autumn,-30
4,2024-09-26 16:20:06.060029000,6.189200,-29.747240,4.200907,39.280730,6.042150,3.872997,True,Autumn,-30
...,...,...,...,...,...,...,...,...,...,...
2702,2025-09-29 21:33:33.133998000,26.282977,-30.085045,5.468698,41.433453,6.354783,3.552399,True,Autumn,-30
2703,2025-09-29 21:45:49.134215000,25.420881,-30.072571,5.592314,41.589166,6.370000,3.609625,True,Autumn,-30
2704,2025-09-29 21:58:05.133970000,26.181910,-30.079753,5.542714,41.523903,6.392333,3.694665,True,Autumn,-30
2705,2025-09-29 22:10:21.133372999,19.554389,-30.081190,5.423461,41.380286,6.389517,3.211531,True,Autumn,-30


In [3]:
import pandas as pd
import pingouin as pg
import numpy as np

df = pd.read_csv(r"D:\Coding\Data\Lanzhou_cfdc\processed\N_INP(202409-202509)v2.4.2.csv")
#df = df[df['Is_Significant'] == True]
target_temp = -30
df_temp = df[df['T_a(degC)'] == target_temp]

# --- 步骤 1：检验方差齐性 (Levene's Test) ---
# 这一步是为了验证你的观察，确认方差确实不齐
levene_test = pg.homoscedasticity(data=df_temp, dv='N_INP(#/L)', group='Season')
print("=== 方差齐性检验 (Levene's Test) ===")
print(levene_test)
print("注：如果 pval < 0.05，说明方差不齐，必须使用 Welch's ANOVA。\n")

# --- 步骤 2：Welch's ANOVA (适用于方差不齐的情况) ---
welch_anova = pg.welch_anova(dv='N_INP(#/L)', between='Season', data=df_temp)
print("=== Welch's ANOVA 结果 ===")
print(welch_anova)
print("注：如果 p-unc < 0.05，说明四个季节的总体平均值至少有两个存在显著差异。\n")

# --- 步骤 3：事后检验 (Games-Howell Test) ---
# 如果 Welch's ANOVA 显著，我们需要知道具体是哪两个季节不同
# Games-Howell 是专门针对方差不齐和样本量可能不等的两两比较方法
if welch_anova['p_unc'].values[0] < 0.05:
    posthoc = pg.pairwise_gameshowell(dv='N_INP(#/L)', between='Season', data=df_temp)
    print("=== 两两比较事后检验 (Games-Howell) ===")
    print(posthoc[['A', 'B', 'mean_A', 'mean_B', 'diff', 'pval']])
    print("注：pval < 0.05 表示这两个季节之间差异显著。")

=== 方差齐性检验 (Levene's Test) ===
                W          pval  equal_var
levene  73.914653  2.991286e-44      False
注：如果 pval < 0.05，说明方差不齐，必须使用 Welch's ANOVA。

=== Welch's ANOVA 结果 ===
   Source  ddof1       ddof2           F         p_unc       np2
0  Season      3  525.941519  195.354142  4.118862e-85  0.159385
注：如果 p-unc < 0.05，说明四个季节的总体平均值至少有两个存在显著差异。

=== 两两比较事后检验 (Games-Howell) ===
        A       B     mean_A     mean_B       diff          pval
0  Autumn  Spring  51.633631  84.004129 -32.370498  5.359032e-07
1  Autumn  Summer  51.633631  23.410065  28.223566  1.027361e-09
2  Autumn  Winter  51.633631   5.695179  45.938452  6.261658e-14
3  Spring  Summer  84.004129  23.410065  60.594064  2.131628e-13
4  Spring  Winter  84.004129   5.695179  78.308950  0.000000e+00
5  Summer  Winter  23.410065   5.695179  17.714886  0.000000e+00
注：pval < 0.05 表示这两个季节之间差异显著。
